<a href="https://colab.research.google.com/github/J4m331/COMP3608-IS-Project/blob/main/Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

In [ ]:
import os

try:
    from google.colab import drive
    os.system("git clone https://github.com/J4m331/COMP3608-IS-Project")
    BASE_DIR = "/content/COMP3608-IS-Project"
except ImportError:
    BASE_DIR = os.path.dirname(os.path.abspath("GA.ipynb"))

#Dataset1/ df1 is the combination of 2 datasets from 2024
Df1 = pd.read_csv(os.path.join(BASE_DIR, "data", "vgData2024.csv"))
Df2 = pd.read_csv(os.path.join(BASE_DIR, "data", "vgData2026.csv"))


In [ ]:
Columns_needed= ["Name","Publisher","Platform","Genre","Release_Year","NA_Sales","EU_Sales","JP_Sales","Other_Sales"]

Df1 = Df1[Columns_needed]
Df2 = Df2[Columns_needed]

print("Dataset 1 after choosing the necessary columns")
print(Df1.head())
print("Dataset 2 after choosing the necessary columns")
print(Df2.head())



In [ ]:
Df1= Df1.drop_duplicates(subset='Name')
Df2= Df2.drop_duplicates(subset='Name')

#Fix release year
Df1['Release_Year'] = pd.to_datetime(Df1['Release_Year'], errors='coerce').dt.year
Df2['Release_Year'] = pd.to_datetime(Df2['Release_Year'], errors='coerce').dt.year

#Fill missing years with median
Df1['Release_Year'] = Df1['Release_Year'].fillna(Df1['Release_Year'].median())
Df2['Release_Year'] = Df2['Release_Year'].fillna(Df2['Release_Year'].median())

# Convert to integer
Df1['Release_Year'] = Df1['Release_Year'].astype(int)
Df2['Release_Year'] = Df2['Release_Year'].astype(int)


print("Dataset 1 after cleaning")
print(Df1.head())
print("Dataset 2 after cleaning")
print(Df2.head())



In [ ]:
#Calculate total sales
Df1['Total_Sales'] = (Df1['NA_Sales']+  Df1['EU_Sales']+Df1['JP_Sales']+Df1['Other_Sales'])
Df2['Total_Sales'] = (Df2['NA_Sales']+  Df2['EU_Sales']+Df2['JP_Sales']+Df2['Other_Sales'])

print("Dataset 1 after calculating total sales")
print(Df1.head())
print("Dataset 2 after calculating total sales")
print(Df2.head())

In [ ]:
#Create sales category
low_threshold1 = Df1['Total_Sales'].quantile(0.33)
high_threshold1 = Df1['Total_Sales'].quantile(0.66)

low_threshold2 = Df2['Total_Sales'].quantile(0.33)
high_threshold2 = Df2['Total_Sales'].quantile(0.66)

def classify_sales1(x):
    if x <= low_threshold1:
        return 'Low'
    elif x <= high_threshold1:
        return 'Medium'
    else:
        return 'High'


def classify_sales2(x):
    if x <= low_threshold2:
        return 'Low'
    elif x <= high_threshold2:
        return 'Medium'
    else:
        return 'High'

# Apply the classification functions to create the 'Sales_Category' column
Df1['Sales_Category'] = Df1['Total_Sales'].apply(classify_sales1)
Df2['Sales_Category'] = Df2['Total_Sales'].apply(classify_sales2)

print("Df1 Sales Categories")

Df1_counts = Df1['Sales_Category'].value_counts()
Df1_percent = (Df1['Sales_Category'].value_counts(normalize=True) * 100).round(2)

Df1_summary = pd.DataFrame({
    'Count': Df1_counts,
    'Percentage': Df1_percent
})

print(Df1_summary)


print("\nDf2 Sales Categories")

Df2_counts = Df2['Sales_Category'].value_counts()
Df2_percent = (Df2['Sales_Category'].value_counts(normalize=True) * 100).round(2)

Df2_summary = pd.DataFrame({
    'Count': Df2_counts,
    'Percentage': Df2_percent
})

print(Df2_summary)

In [ ]:
#Encode categorical variables

le_genre = LabelEncoder()
le_platform = LabelEncoder()
le_publisher = LabelEncoder()
le_target = LabelEncoder()

Df1['Genre_enc'] = le_genre.fit_transform(Df1['Genre'])
Df1['Platform_enc'] = le_platform.fit_transform(Df1['Platform'])
Df1['Publisher_enc'] = le_publisher.fit_transform(Df1['Publisher'])
Df1['Sales_Category_enc'] = le_target.fit_transform(Df1['Sales_Category'])

Df2['Genre_enc'] = le_genre.transform(Df2['Genre'])
Df2['Platform_enc'] = le_platform.transform(Df2['Platform'])
Df2['Publisher_enc'] = le_publisher.transform(Df2['Publisher'])
Df2['Sales_Category_enc'] = le_target.transform(Df2['Sales_Category'])

print("Df1 Before vs After Encoding")

print(Df1[['Genre', 'Genre_enc',
           'Platform', 'Platform_enc',
           'Publisher', 'Publisher_enc',
           'Sales_Category', 'Sales_Category_enc']].head())


print("\nDf2 Before vs After Encoding")

print(Df2[['Genre', 'Genre_enc',
           'Platform', 'Platform_enc',
           'Publisher', 'Publisher_enc',
           'Sales_Category', 'Sales_Category_enc']].head())

In [ ]:
#Features and target
X1 = Df1[['Genre_enc', 'Platform_enc', 'Publisher_enc', 'Release_Year']]
Y2 = Df1['Sales_Category_enc']

X2 = Df2[['Genre_enc', 'Platform_enc', 'Publisher_enc', 'Release_Year']]
Y2 = Df2['Sales_Category_enc']

In [ ]:
X1_train, X1_test, Y1_train, Y1_test= train_test_split(X1, Y2, test_size=0.3, random_state=100)
X2_train, X2_test, Y2_train, Y2_test= train_test_split(X2, Y2, test_size=0.3, random_state=100)


In [ ]:
model2 = DecisionTreeClassifier(max_depth = 5, random_state = 100)
model2.fit(X2_train, Y2_train)

In [ ]:
model1 = DecisionTreeClassifier(max_depth = 5, random_state = 100)
model1.fit(X1_train, Y1_train)



In [ ]:
print("Df1 metrics")
y1_pred = model1.predict(X1_test)
accuracy1 = accuracy_score(Y1_test, y1_pred)
print(f"Accuracy:{accuracy1:.4f}")
print("\nClssification Report:\n",classification_report(Y1_test, y1_pred))
print("\nConfusion Matrix:\n",confusion_matrix(Y1_test, y1_pred))

print("\n")

print("Df2 metrics")
y2_pred = model2.predict(X2_test)
accuracy2 = accuracy_score(Y2_test, y2_pred)
print(f"Accuracy:{accuracy2:.4f}")
print("\nClssification Report:\n",classification_report(Y2_test, y2_pred))
print("\nConfusion Matrix:\n",confusion_matrix(Y2_test, y2_pred))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy1 = accuracy_score(Y1_test, y1_pred)
precision1 = precision_score(Y1_test, y1_pred, average='weighted')
recall1 = recall_score(Y1_test, y1_pred, average='weighted')
f1score1 = f1_score(Y1_test, y1_pred, average='weighted')

accuracy2 = accuracy_score(Y2_test, y2_pred)
precision2 = precision_score(Y2_test, y2_pred, average='weighted')
recall2 = recall_score(Y2_test, y2_pred, average='weighted')
f1score2 = f1_score(Y2_test, y2_pred, average='weighted')

results = pd.DataFrame({
    'Dataset': ['Dataset1', 'Dataset2'],
    'Accuracy': [accuracy1, accuracy2],
    'Precision': [precision1, precision2],
    'Recall': [recall1, recall2],
    'F1 Score': [f1score1, f1score2]
})

print(results)

In [ ]:
#Bar chart of metrics
results.plot(x='Dataset',y=['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    kind='bar',figsize=(10,6))

plt.title("Performance Comparison of Datasets")
plt.ylabel("Score")
plt.xlabel("Dataset")
plt.ylim(0,1)

plt.show()

In [ ]:
#Feature importance

importance1 = pd.DataFrame({'Feature': X1.columns,'Importance': model1.feature_importances_})
importance1 = importance1.sort_values(by='Importance',ascending=False)

importance2 = pd.DataFrame({'Feature': X2.columns,'Importance': model2.feature_importances_})
importance2 = importance2.sort_values(by='Importance',ascending=False)

print("\nDataset 1 Feature Importance:\n")
print(importance1)

print("\nDataset 2 Feature Importance:\n")
print(importance2)



In [ ]:
#Importance graph of Dataset1
plt.figure(figsize=(8,5))

plt.bar(importance1['Feature'],importance1['Importance'])
plt.title('Dataset 1 Feature Importance')
plt.xlabel('Features')
plt.ylabel('Importance Score')

plt.show()

In [ ]:
#Importance graph of Dataset2
plt.figure(figsize=(8,5))

plt.bar(importance2['Feature'],importance1['Importance'])
plt.title('Dataset 2 Feature Importance')
plt.xlabel('Features')
plt.ylabel('Importance Score')

plt.show()

In [ ]:
# Top 10 most profitable companies (publishers) for Dataset 1
top_publishers1 = (
    Df1.groupby("Publisher")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("Top 10 Most Profitable Publishers - Dataset 1")
print(top_publishers1)

# Top 10 most profitable companies (publishers) for Dataset 2
top_publishers2 = (
    Df2.groupby("Publisher")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("\nTop 10 Most Profitable Publishers - Dataset 2")
print(top_publishers2)

# Optional: Plot bar charts for visualization
fig, axes = plt.subplots(1, 2, figsize=(14,6))

axes[0].barh(top_publishers1["Publisher"], top_publishers1["Total_Sales"], color="skyblue")
axes[0].set_title("Top 10 Publishers - Dataset 1")
axes[0].set_xlabel("Total Sales (millions)")
axes[0].invert_yaxis()

axes[1].barh(top_publishers2["Publisher"], top_publishers2["Total_Sales"], color="lightgreen")
axes[1].set_title("Top 10 Publishers - Dataset 2")
axes[1].set_xlabel("Total Sales (millions)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()
